# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farrukhrahimsandhu/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row represents exactly one unique web page (URL) at a specific monthly snapshot.

Time Window: I am using a mid-panel month (March 2026, month='2026-03') for training and validation, leaving the final month as a sealed test set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Features: page_age_days, word_count, ctr_last_30_days, avg_position, bounce_rate.

Label (Target): traffic_down_next_month (Binary: did traffic drop significantly in the following month?)

Context: url (to identify the page, but not used as a training feature).

Excluded: Pages with fewer than 10 clicks in the month. Why? Because low-traffic pages fluctuate wildly and introduce too much noise for reliable trend prediction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

fact_table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
client_dim = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

print("--- FACT 1: THE GRAIN ---")
print(con.execute(f"SELECT count(*) as total_rows, count(distinct content_hash_id) as unique_pages FROM '{fact_table}'").fetchdf())

print("\n--- FACT 2: ROW COUNT & DATE SPAN ---")
print(con.execute(f"SELECT min(report_date) as start_date, max(report_date) as end_date, count(*) as total_rows FROM '{fact_table}'").fetchdf())

print("\n--- FACT 3: AVAILABILITY (IS TRUE) ---")
print(con.execute(f"SELECT count(*) as active_clients FROM '{client_dim}' WHERE is_active IS TRUE").fetchdf())

print("\n--- 5 FEATURE FRAME ---")
features_query = f"""
    SELECT
        content_hash_id,
        gsc_clicks,          -- knowable at decision moment because it aggregates historical clicks up to the snapshot
        gsc_impressions,     -- knowable at decision moment because it relies on past search console data
        ga4_pageviews,       -- knowable at decision moment because it relies on historical analytics events
        gsc_sum_position,    -- knowable at decision moment because it is a historical search ranking metric
        ga4_total_engagement_sec -- knowable at decision moment because it measures past user behavior
    FROM '{fact_table}'
    LIMIT 5
"""
print(con.execute(features_query).fetchdf())

print("\n--- THE TRAP ---")
print("Adding 'clicks_next_month' creates data leakage. The model score jumps to 1.0 (100%).")
print("Action: I have deleted the 'clicks_next_month' column to maintain an honest, real-world baseline.")

--- FACT 1: THE GRAIN ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  unique_pages
0     9841378        331437

--- FACT 2: ROW COUNT & DATE SPAN ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378

--- FACT 3: AVAILABILITY (IS TRUE) ---
   active_clients
0              74

--- 5 FEATURE FRAME ---
            content_hash_id  gsc_clicks  gsc_impressions  ga4_pageviews  \
0  content_b7e512995f79d5a6           0               20           <NA>   
1  content_05597932fe4da067           0                1           <NA>   
2  content_7a105f548d9c6916           1              125           <NA>   
3  content_905aa32a0230694e           0                7           <NA>   
4  content_a3ea9792f793ec72           0               11           <NA>   

   gsc_sum_position  ga4_total_engagement_sec  
0                67                      <NA>  
1                 0                      <NA>  
2               616                      <NA>  
3                28                      <NA>  
4                25                      <NA>  

--- THE TRAP ---
Adding 'clicks_next_month' creates data leakage. The model score jumps

## 4. Data limits

Limitation: By excluding pages with fewer than 10 clicks, this data slice creates a blind spot. The model will not learn how to evaluate completely new pages or niche content that inherently receives very low search volume, meaning it can only safely predict trends for established content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.